<i>Copyright (c) Recommenders contributors.</i>

<i>Licensed under the MIT License.</i>

# Two-Tower (Dual-Encoder) Retrieval on MovieLens

Large recommender systems usually work in two stages: a fast **retrieval** stage selects a few hundred candidates out of the whole catalogue, and a slower **ranking** stage orders them. The Two-Tower model is the standard approach for the retrieval stage [1, 2, 3].

It has two neural networks ("towers"):

* the **user tower** maps a user (ID and, optionally, side features) to a vector,
* the **item tower** maps an item (ID and, optionally, side features such as genres) to a vector in the same space.

The score of a user-item pair is the dot product of their vectors. Since the item vectors don't depend on the user, they can be computed once and stored in a (approximate) nearest-neighbour index, so retrieving the top items for a user is a single vector search, even with millions of items.

### Training objective

The model is trained on implicit feedback (who interacted with what) with an **in-batch sampled softmax**: for each (user, item) pair in a batch, the other items of the batch act as negatives. Popular items appear as negatives more often, so the loss applies the **logQ correction** of Yi et al. [2], subtracting the log of each item's sampling probability from its logit.

In this notebook we train a Two-Tower model on MovieLens, using the movie genres as item features, and evaluate the top-k retrieval quality with ranking metrics.

## 0. Global settings and imports

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import sys
import logging
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd
import torch

from recommenders.utils.constants import (
    DEFAULT_USER_COL as USER,
    DEFAULT_ITEM_COL as ITEM,
    DEFAULT_RATING_COL as RATING,
    DEFAULT_TIMESTAMP_COL as TIMESTAMP,
    DEFAULT_GENRE_COL as GENRE,
    DEFAULT_PREDICTION_COL as PREDICTION,
)
from recommenders.datasets import movielens
from recommenders.datasets.python_splitters import python_stratified_split
from recommenders.evaluation.python_evaluation import (
    map_at_k,
    ndcg_at_k,
    precision_at_k,
    recall_at_k,
)
from recommenders.models.two_tower.data_loader import TwoTowerData
from recommenders.models.two_tower.model import TwoTowerModel
from recommenders.models.two_tower.training_utils import Trainer
from recommenders.models.two_tower.utils import multi_hot_features, recommend_k_items
from recommenders.utils.notebook_utils import store_metadata
from recommenders.utils.timer import Timer

logging.basicConfig(level=logging.INFO, format="%(message)s")

print(f"System version: {sys.version}")
print(f"Pandas version: {pd.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

System version: 3.11.18 (main, Oct 26 2026, 15:28:55) [GCC 13.3.0]
Pandas version: 2.3.3
PyTorch version: 2.15.0+cu132
CUDA available: False


In [2]:
# Top k items to recommend
TOP_K = 10

# Select MovieLens data size: 100k, 1m, 10m, or 20m
MOVIELENS_DATA_SIZE = "100k"

# Model parameters
EMBEDDING_DIM = 64
HIDDEN_UNITS = [128]
OUTPUT_DIM = 64
USE_ITEM_FEATURES = True

# Training parameters
EPOCHS = 5
BATCH_SIZE = 1024
LEARNING_RATE = 1e-3
TEMPERATURE = 0.05
SEED = 42

## 1. Load and split the data

We load the ratings together with the movie genres. Every rating is treated as an implicit positive interaction: the model learns which movies a user watches, not how they rate them.

In [3]:
df = movielens.load_pandas_df(
    size=MOVIELENS_DATA_SIZE,
    header=[USER, ITEM, RATING, TIMESTAMP],
    genres_col=GENRE,
)
df.head()

,userID,itemID,rating,timestamp,genre
0,196,242,3.0,881250949,Comedy
1,186,302,3.0,891717742,Crime|Film-Noir|Mystery|Thriller
2,22,377,1.0,878887116,Children's|Comedy
3,244,51,2.0,880606923,Drama|Romance|War|Western
4,166,346,1.0,886397596,Crime|Drama


In [4]:
train, test = python_stratified_split(
    df, ratio=0.75, col_user=USER, col_item=ITEM, seed=SEED
)

# Remove the items that only appear in the test set, since the model can't retrieve them
test = test[test[ITEM].isin(train[ITEM])]

print(f"Train: {len(train)} interactions, {train[USER].nunique()} users, {train[ITEM].nunique()} items")
print(f"Test:  {len(test)} interactions, {test[USER].nunique()} users, {test[ITEM].nunique()} items")

Train: 74992 interactions, 943 users, 1653 items
Test:  24973 interactions, 943 users, 1415 items


## 2. Prepare the item features and the training data

The item tower receives the genres of each movie as a multi-hot vector. This lets movies that share genres start close to each other in the vector space, which helps items with few interactions.

`TwoTowerData` maps the user and item IDs to indices, keeps the item frequencies for the logQ correction, aligns the feature matrix with the item indices and remembers which items each user has seen.

In [5]:
item_features = multi_hot_features(df, col_id=ITEM, col_values=GENRE) if USE_ITEM_FEATURES else None
if item_features is not None:
    print(f"{item_features.shape[1] - 1} genre features")

data = TwoTowerData(train, col_user=USER, col_item=ITEM, item_features=item_features)
print(f"{data.n_users} users, {data.n_items} items")

19 genre features
943 users, 1653 items


## 3. Train the model

In [6]:
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

model = TwoTowerModel(
    n_users=data.n_users,
    n_items=data.n_items,
    embedding_dim=EMBEDDING_DIM,
    hidden_units=HIDDEN_UNITS,
    output_dim=OUTPUT_DIM,
    item_features=data.item_features,
)
print(model)

TwoTowerModel(
  (user_tower): Tower(
    (embedding): Embedding(943, 64)
    (mlp): Sequential(
      (0): Linear(in_features=64, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=64, bias=True)
    )
  )
  (item_tower): Tower(
    (embedding): Embedding(1653, 64)
    (mlp): Sequential(
      (0): Linear(in_features=83, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=64, bias=True)
    )
  )
)


The trainer uses Adam and the in-batch softmax loss. Passing `item_counts` turns on the logQ correction. Larger batches give more negatives for each positive, which usually improves retrieval quality.

In [7]:
trainer = Trainer(
    model,
    learning_rate=LEARNING_RATE,
    temperature=TEMPERATURE,
    item_counts=data.item_counts,
)
train_dl = data.loader(batch_size=BATCH_SIZE, shuffle=True, seed=SEED)

with Timer() as train_time:
    history = trainer.fit(train_dl, n_epochs=EPOCHS)

print(f"Took {train_time.interval:.2f} seconds for training.")
pd.DataFrame(history).tail()

epoch=1, train_loss=7.1349


epoch=2, train_loss=6.8198


epoch=3, train_loss=6.6334


epoch=4, train_loss=6.4746


epoch=5, train_loss=6.4051


Took 6.22 seconds for training.


,epoch,train_loss
0,1,7.134923
1,2,6.819824
2,3,6.633404
3,4,6.474572
4,5,6.405082


The model can be saved and loaded back with the usual PyTorch state dict. The item features are part of it.

In [8]:
tmp = TemporaryDirectory()
model_path = os.path.join(tmp.name, "two_tower.pth")
torch.save(model.state_dict(), model_path)

loaded_model = TwoTowerModel(
    n_users=data.n_users,
    n_items=data.n_items,
    embedding_dim=EMBEDDING_DIM,
    hidden_units=HIDDEN_UNITS,
    output_dim=OUTPUT_DIM,
    item_features=data.item_features,
)
loaded_model.load_state_dict(torch.load(model_path, map_location="cpu"))
loaded_model.to(trainer.device)
loaded_model.eval()
print("Model loaded successfully.")

Model loaded successfully.


## 4. Retrieve the top-k items

`recommend_k_items` encodes all items once, scores them against each user vector and keeps the `TOP_K` best items the user hasn't interacted with in the training data. In production, the vectors from `model.encode_items()` would go into an approximate nearest-neighbour index instead.

In [9]:
with Timer() as test_time:
    top_k = recommend_k_items(
        loaded_model,
        data,
        users=test[USER].unique(),
        top_k=TOP_K,
        remove_seen=True,
        col_prediction=PREDICTION,
    )

print(f"Took {test_time.interval:.2f} seconds for retrieval.")
top_k.head(TOP_K)

Took 0.01 seconds for retrieval.


,userID,itemID,prediction
0,1,204,0.004137
1,1,288,0.001940
2,1,318,-0.001676
3,1,258,-0.003207
4,1,181,-0.005132
5,1,423,-0.006627
6,1,474,-0.007392
7,1,174,-0.009301
8,1,121,-0.010550
9,1,655,-0.012988


## 5. Evaluate

In [10]:
eval_map = map_at_k(test, top_k, col_user=USER, col_item=ITEM, col_rating=RATING, col_prediction=PREDICTION, k=TOP_K)
eval_ndcg = ndcg_at_k(test, top_k, col_user=USER, col_item=ITEM, col_rating=RATING, col_prediction=PREDICTION, k=TOP_K)
eval_precision = precision_at_k(test, top_k, col_user=USER, col_item=ITEM, col_rating=RATING, col_prediction=PREDICTION, k=TOP_K)
eval_recall = recall_at_k(test, top_k, col_user=USER, col_item=ITEM, col_rating=RATING, col_prediction=PREDICTION, k=TOP_K)

print(
    "Model:\t\t" + model.__class__.__name__,
    "Top K:\t\t%d" % TOP_K,
    "MAP@K:\t\t%f" % eval_map,
    "NDCG@K:\t\t%f" % eval_ndcg,
    "Precision@K:\t%f" % eval_precision,
    "Recall@K:\t%f" % eval_recall,
    sep="\n",
)

Model:		TwoTowerModel
Top K:		10
MAP@K:		0.271784
NDCG@K:		0.411537
Precision@K:	0.368187
Recall@K:	0.192906


In [11]:
# Record results for tests - ignore this cell
store_metadata("map", eval_map)
store_metadata("ndcg", eval_ndcg)
store_metadata("precision", eval_precision)
store_metadata("recall", eval_recall)
store_metadata("train_time", train_time.interval)
store_metadata("test_time", test_time.interval)

In [12]:
tmp.cleanup()

## References

1. Po-Sen Huang, Xiaodong He, Jianfeng Gao, Li Deng, Alex Acero and Larry Heck, "Learning Deep Structured Semantic Models for Web Search using Clickthrough Data", CIKM 2013.
2. Xinyang Yi, Ji Yang, Lichan Hong, Derek Zhiyuan Cheng, Lukasz Heldt, Aditee Kumthekar, Zhe Zhao, Li Wei and Ed Chi, "Sampling-Bias-Corrected Neural Modeling for Large Corpus Item Recommendations", RecSys 2019.
3. Paul Covington, Jay Adams and Emre Sargin, "Deep Neural Networks for YouTube Recommendations", RecSys 2016.